## Imports and Supabase Connection

In [1]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
from pathlib import Path
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

%matplotlib inline
# Resolve the project root (one level above `/notebooks`).
project_root = Path.cwd().parent
env_path = project_root / ".env"

load_dotenv(dotenv_path=env_path, override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Optional sanity check.
print("Loaded key starts with:", SUPABASE_KEY[:5])


Loaded key starts with: eyJhb


## Incident

In [2]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("incident")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

incident_df = pd.DataFrame(all_rows)
incident_df["Number_Victims"] = pd.to_numeric(incident_df["Number_Victims"], errors="coerce")

# Quick validation of total row count.
print (len(incident_df))
print (incident_df.head(2))


3136
     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   

                             School Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School              0               1   
1                Carver High School              1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        Yes   

  Security_Screening     Screening_Outcome Shots_Fired School_Lockdown  \
0                                                    7                   
1       Armed Guards  Outside/Off-Property           3                   

         LAT         LNG Campus_Type Zipcode  
0  35.237069  -80.850227               28202  
1   31.57954  -97.130303               76704  

[2 rows x 50 columns]


## Background Checks

In [3]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("background_check_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

background_check_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(background_check_df))
print (background_check_df.head(2))


21960
     state state_abb  fips  year year_frac        law_class law_class_subtype  \
0  Alabama        AL     1  1979         1  Castle doctrine               N/A   
1  Alabama        AL     1  1979         1   Dealer license               N/A   

  handguns_or_long_guns age_for_minimum_age_laws  law_id  
0  handgun and long gun                     #N/A  AL1004  
1               handgun                     #N/A  AL1007  


# Model Exploring

## Incidents

In [4]:
incident_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3136 entries, 0 to 3135
Data columns (total 50 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Incident_ID              3136 non-null   str  
 1   Month                    3136 non-null   int64
 2   Day                      3136 non-null   int64
 3   Year                     3136 non-null   int64
 4   Date                     3136 non-null   str  
 5   School                   3136 non-null   str  
 6   Victims_Killed           3136 non-null   str  
 7   Victims_Wounded          3136 non-null   str  
 8   Number_Victims           3136 non-null   int64
 9   Shooter_Killed           3136 non-null   str  
 10  Source                   3136 non-null   str  
 11  Number_News              3136 non-null   str  
 12  Media_Attention          3136 non-null   str  
 13  Reliability              3136 non-null   str  
 14  Quarter                  3136 non-null   str  
 15  City           

### Apply the best subset selection approach to incident data. We wish to predict risk based on a variety of features/predictors

In [5]:
# row count for incident table
incident_df.shape

(3136, 50)

In [6]:
# year range for incident table
incident_df['Year'].min(), incident_df['Year'].max()

(np.int64(1966), np.int64(2025))

In [7]:
# print columns for incident table
incident_df.columns

Index(['Incident_ID', 'Month', 'Day', 'Year', 'Date', 'School',
       'Victims_Killed', 'Victims_Wounded', 'Number_Victims', 'Shooter_Killed',
       'Source', 'Number_News', 'Media_Attention', 'Reliability', 'Quarter',
       'City', 'State', 'School_Level', 'Location', 'Location_Type',
       'During_Classes', 'Time_Period', 'First_Shot', 'Duration_min',
       'Summary', 'Narrative', 'Situation', 'GV_Type',
       'Involves_Students_Staff', 'Targets', 'Accomplice',
       'Accomplice_Narrative', 'Hostages', 'Barricade', 'Officer_Involved',
       'Bullied', 'Domestic_Violence', 'Gang_Related', 'Active_Shooter_FBI',
       'Multiple_Location', 'Preplanned', 'SRO_School', 'Security_Screening',
       'Screening_Outcome', 'Shots_Fired', 'School_Lockdown', 'LAT', 'LNG',
       'Campus_Type', 'Zipcode'],
      dtype='str')

In [8]:
# number of missing values in Victim_Killed column
incident_df['Victims_Killed'].isna().sum()

np.int64(0)

In [9]:
# number of unique states in incident table
incident_df['State'].unique()

<StringArray>
[                 'NC',                  'TX',                  'CA',
                  'NY',                  'FL',                  'PA',
                  'MN',                  'TN',                  'IL',
                  'VT',                  'HI',                  'OK',
                  'KY',                  'DC',                  'WI',
                  'OH',                  'AR',                  'DE',
                  'UT',                  'ID',                  'IA',
                  'MI',              'Winter',                  'NJ',
                  'AZ',                  'VA',                  'MD',
                  'NM',                  'LA',                  'NV',
                  'IN',             'Houston',                  'AL',
                  'MO',                  'CT',         'Rogersville',
           'Hahnville',                  'SC',             'Artesia',
                  'GA',           'Pensacola',           'St. Louis',
      

In [10]:
valid_states = {
"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA",
"HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
"MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
"NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
"SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

invalid = incident_df[~incident_df["State"].isin(valid_states)]
invalid["State"].unique()

<StringArray>
[             'Winter',             'Houston',         'Rogersville',
           'Hahnville',             'Artesia',           'Pensacola',
           'St. Louis',              'Dallas',          'Fort Worth',
              'Spring',                'Fall',            'Hartford',
            'Anniston',               'Miami',              'Aldine',
            'Portland',        'New Rochelle',        'Indianapolis',
               'Tampa',             'Bristol',          'Fort Myers',
            'Winnetka',             'Detroit',                'Oahu',
         'Baton Rouge',         'Port Arthur',      'Virginia Beach',
            'Stockton',          'Washington',             'Slidell',
           'Rock Hill',              'Merced',            'Amarillo',
          'Montgomery',                   '4',              'Summer',
 'San Fernando Valley',          'Manchester',            'New York',
               'Omaha',                  'VI',                    '',
      

In [11]:
incident_df[incident_df["State"] == "Houston"].head()

,Incident_ID,Month,Day,Year,Date,School,Victims_Killed,Victims_Wounded,Number_Victims,Shooter_Killed,...,Preplanned,SRO_School,Security_Screening,Screening_Outcome,Shots_Fired,School_Lockdown,LAT,LNG,Campus_Type,Zipcode
83,19720914TXFRH,9,14,1972,1972-09-14T00:00:00+00:00,Francis Scott Key Junior High,0,1,1,0,...,,No,,Unknown,NA,<10,,29.812189,-95.330081,
85,19720919TXCUH,9,19,1972,1972-09-19T00:00:00+00:00,Cullen Junior High,0,1,1,0,...,,No,,Unknown,NA,9,,29.691239,-95.36387,
253,19830118TXJOH,1,18,1983,1983-01-18T00:00:00+00:00,John H. Reagan High School,1,0,1,1,...,,No,,Unknown,NA,4,,29.795407,-95.39328,
498,19920930TXHOH,9,30,1992,1992-09-30T00:00:00+00:00,Hollibrook Elementary School,1,1,2,0,...,,No,,Unknown,NA,2,,29.826246,-95.50733,


In [12]:
incident_df_from_1987 = incident_df[incident_df["Year"] >= 1987]
# year range for incident table
incident_df_from_1987['Year'].min(), incident_df_from_1987['Year'].max()

(np.int64(1987), np.int64(2025))

In [13]:
incident_df_from_1987.columns

Index(['Incident_ID', 'Month', 'Day', 'Year', 'Date', 'School',
       'Victims_Killed', 'Victims_Wounded', 'Number_Victims', 'Shooter_Killed',
       'Source', 'Number_News', 'Media_Attention', 'Reliability', 'Quarter',
       'City', 'State', 'School_Level', 'Location', 'Location_Type',
       'During_Classes', 'Time_Period', 'First_Shot', 'Duration_min',
       'Summary', 'Narrative', 'Situation', 'GV_Type',
       'Involves_Students_Staff', 'Targets', 'Accomplice',
       'Accomplice_Narrative', 'Hostages', 'Barricade', 'Officer_Involved',
       'Bullied', 'Domestic_Violence', 'Gang_Related', 'Active_Shooter_FBI',
       'Multiple_Location', 'Preplanned', 'SRO_School', 'Security_Screening',
       'Screening_Outcome', 'Shots_Fired', 'School_Lockdown', 'LAT', 'LNG',
       'Campus_Type', 'Zipcode'],
      dtype='str')

In [14]:
# number of unique states in incident table
incident_df_from_1987['State'].unique()

<StringArray>
[                 'CA',                  'MI',                  'AR',
                  'AZ',        'New Rochelle',                  'MO',
                  'TX',              'Spring',        'Indianapolis',
                  'FL',                  'IL',                  'NC',
                  'SC',              'Winter',                  'NY',
                  'PA',               'Tampa',                  'LA',
                  'AL',             'Bristol',          'Fort Myers',
            'Winnetka',                  'MS',             'Detroit',
                'Oahu',                  'WI',                  'GA',
         'Baton Rouge',                  'MD',                'Fall',
         'Port Arthur',                  'UT',      'Virginia Beach',
            'Stockton',          'Washington',                  'ID',
                  'IN',              'Dallas',                  'VA',
                  'KY',                  'RI',                  'OH',
      

In [15]:
analytic_df = incident_df[(incident_df["Year"] >= 1987) & (incident_df["Year"] <= 2023)]
invalid_analytic = analytic_df[~analytic_df["State"].isin(valid_states)]
invalid_analytic.shape

(46, 50)

In [16]:
analytic_df.shape

(2311, 50)

## Sample Restriction (1987–Present)

The panel is restricted to 1987 onward due to a structural break in the underlying data-generating process prior to 1987.

Pre-1987 observations are not comparable in measurement, reporting coverage, and policy codification. Including those years would violate panel homogeneity assumptions and introduce non-structural noise into fixed effects and dynamic specifications.

This restriction is not outcome-based filtering. It is a data consistency adjustment to ensure:

- Comparable incident classification
- Consistent enrollment coverage
- Reliable policy coding
- Stable variance structure

All models are therefore estimated on the 1987–present balanced policy-enrollment panel.


## Incidents from 1987

In [17]:
incident_df_from_1987 = incident_df[incident_df["State"].isin(valid_states)]
print(incident_df_from_1987.shape) # incident_df_from_1987.shape
print(incident_df_from_1987["State"].unique())

(3070, 50)
<StringArray>
['NC', 'TX', 'CA', 'NY', 'FL', 'PA', 'MN', 'TN', 'IL', 'VT', 'HI', 'OK', 'KY',
 'DC', 'WI', 'OH', 'AR', 'DE', 'UT', 'ID', 'IA', 'MI', 'NJ', 'AZ', 'VA', 'MD',
 'NM', 'LA', 'NV', 'IN', 'AL', 'MO', 'CT', 'SC', 'GA', 'CO', 'MA', 'WV', 'MS',
 'OR', 'MT', 'KS', 'WA', 'WY', 'RI', 'NH', 'NE', 'AK', 'ME', 'SD', 'ND']
Length: 51, dtype: str


In [18]:
# incident count per state-year combination
incident_counts = (
    incident_df_from_1987
    .groupby(["State", "Year"])
    .size()
    .reset_index(name="incident_count")
)

# Ensure Victims_Killed is numeric before aggregation
incident_df_from_1987["Victims_Killed"] = pd.to_numeric(
    incident_df_from_1987["Victims_Killed"],
    errors="coerce"
)

# Replace any non-numeric values with 0
incident_df_from_1987["Victims_Killed"] = (
    incident_df_from_1987["Victims_Killed"].fillna(0)
)

In [19]:
# total victims killed per state-year combination
total_victims = (
    incident_df_from_1987
    .groupby(["State", "Year"])
    .agg({"Victims_Killed": "sum"})
    .reset_index()
)

total_victims

,State,Year,Victims_Killed
0,AK,1997,2
1,AK,2005,0
2,AK,2018,0
3,AK,2019,0
4,AK,2022,0
...,...,...,...
1084,WV,2022,0
1085,WV,2023,0
1086,WV,2024,0
1087,WY,1986,0


In [20]:
# total number of missing years
total_victims.isna().sum()


State             0
Year              0
Victims_Killed    0
dtype: int64

In [21]:
# supabase.table("state_year_enrollment").select("*").execute()

In [22]:
response = supabase.table("Enrollment Data Final").select("*").limit(1).execute(); print(response.data)

[{'school_id': 601195, 'state': 'California', 'year': 1996, 'total_students': 154}]


In [23]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("enrollment_state_year_mat")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

enrollment_df = pd.DataFrame(all_rows)

print(enrollment_df.shape)

(2548, 3)


In [24]:
print(response)
print(len(response.data))

data=[] count=None
0


In [25]:
# Keep only incident years that exist in enrollment data
incident_counts = incident_counts[
    incident_counts["Year"] >= 1987
]

In [26]:
enrollment_df


,state,year,total_students
0,RHODE ISLAND,2020,10006
1,Louisiana,2025,569440
2,Montana,1999,160294
3,Georgia,2015,1721666
4,Rhode Island,2016,139409
...,...,...,...
2543,District of Columbia,2013,75819
2544,Kentucky,2014,683957
2545,GEORGIA,2020,87127
2546,DISTRICT OF COLUMBIA,2015,1671


In [27]:
# Convert full state names to uppercase for consistent mapping
enrollment_df["state"] = enrollment_df["state"].str.upper()


# Map full state names to two-letter abbreviations
state_abbrev = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT',
    'DELAWARE': 'DE', 'DISTRICT OF COLUMBIA': 'DC', 'FLORIDA': 'FL',
    'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL',
    'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS', 'KENTUCKY': 'KY',
    'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS', 'MISSOURI': 'MO', 'MONTANA': 'MT',
    'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH',
    'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA',
    'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX',
    'UTAH': 'UT', 'VERMONT': 'VT', 'VIRGINIA': 'VA',
    'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY'
}

# Create State column matching incident dataset
enrollment_df["State"] = enrollment_df["state"].map(state_abbrev)

# Rename year column for merge consistency
enrollment_df = enrollment_df.rename(columns={"year": "Year"})

In [28]:
# Keep only the larger enrollment value per state–year
enrollment_df = (
    enrollment_df
    .sort_values("total_students", ascending=False)
    .drop_duplicates(["State", "Year"])
)

In [29]:
# Merge incident counts with enrollment data on State and Year
merged_df_table = incident_counts.merge(
    enrollment_df[["State", "Year", "total_students"]],
    on=["State", "Year"],
    how="left"
)

# Check for any unmatched rows after merge
print(merged_df_table["total_students"].isna().sum())

# Compute incident rate per 100,000 students
merged_df_table["incident_rate_per_100k"] = (
    merged_df_table["incident_count"] / merged_df_table["total_students"]
) * 100000

# Inspect result
print(merged_df_table.head())

0
  State  Year  incident_count  total_students  incident_rate_per_100k
0    AK  1997               1          129919                0.769710
1    AK  2005               1          131701                0.759296
2    AK  2018               1          129660                0.771248
3    AK  2019               1          127641                0.783447
4    AK  2022               1          126457                0.790783


In [30]:
print(incident_counts["Year"].min(), incident_counts["Year"].max())
print(enrollment_df["Year"].min(), enrollment_df["Year"].max())

1987 2025
1987 2025


## Correlations 

### background check correlation

In [31]:
background_check_df.columns

Index(['state', 'state_abb', 'fips', 'year', 'year_frac', 'law_class',
       'law_class_subtype', 'handguns_or_long_guns',
       'age_for_minimum_age_laws', 'law_id'],
      dtype='str')

In [32]:
# Rename columns to match merged_df
background_check_df = background_check_df.rename(
    columns={
        "state_abb": "State",
        "year": "Year"
    }
)

# Ensure correct format
background_check_df["State"] = background_check_df["State"].str.upper()
background_check_df["Year"] = background_check_df["Year"].astype(int)

In [33]:
# Create binary state–year indicator for background check laws
background_indicator = (
    background_check_df
    [background_check_df["law_class"] == "Background checks"]
    .groupby(["State", "Year"])
    .size()
    .reset_index(name="background_check_law")
)

# Set indicator to 1
background_indicator["background_check_law"] = 1

In [34]:
# Keep only one row per State–Year
merged_df_table = merged_df_table.drop_duplicates(["State", "Year"])

In [35]:
# Keep only one row per State–Year
merged_df_table = merged_df_table.drop_duplicates(["State", "Year"])

In [36]:
len(merged_df_table)

861

In [37]:
print(incident_df.columns.tolist())

['Incident_ID', 'Month', 'Day', 'Year', 'Date', 'School', 'Victims_Killed', 'Victims_Wounded', 'Number_Victims', 'Shooter_Killed', 'Source', 'Number_News', 'Media_Attention', 'Reliability', 'Quarter', 'City', 'State', 'School_Level', 'Location', 'Location_Type', 'During_Classes', 'Time_Period', 'First_Shot', 'Duration_min', 'Summary', 'Narrative', 'Situation', 'GV_Type', 'Involves_Students_Staff', 'Targets', 'Accomplice', 'Accomplice_Narrative', 'Hostages', 'Barricade', 'Officer_Involved', 'Bullied', 'Domestic_Violence', 'Gang_Related', 'Active_Shooter_FBI', 'Multiple_Location', 'Preplanned', 'SRO_School', 'Security_Screening', 'Screening_Outcome', 'Shots_Fired', 'School_Lockdown', 'LAT', 'LNG', 'Campus_Type', 'Zipcode']


At state-year level, severity should not be modeled per incident.

In [38]:
incident_df_from_1987 = incident_df_from_1987[incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)]
print("Remaining rows:", len(incident_df_from_1987))

Remaining rows: 3070


In [39]:
valid_states = incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)
print("Valid rows:", valid_states.sum())
print("Invalid rows:", (~valid_states).sum())

Valid rows: 3070
Invalid rows: 0


In [40]:
print(incident_df_from_1987.head())

     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   
2  19660324CACAM      3   24  1966  1966-03-24T00:00:00+00:00   
3  19660328CAJOL      3   28  1966  1966-03-28T00:00:00+00:00   
4  19660427NYBAB      4   27  1966  1966-04-27T00:00:00+00:00   

                             School  Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School               0               1   
1                Carver High School               1               0   
2    Camino Pablo Elementary School               3               0   
3                Jordan High School               0               1   
4             Bay Shore High School               1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        

In [41]:
print(merged_df_table["incident_count"].describe())

count    861.000000
mean       3.191638
std        3.745599
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       25.000000
Name: incident_count, dtype: float64


In [42]:
print(merged_df_table["total_students"].describe())

count    8.610000e+02
mean     1.339581e+06
std      1.313322e+06
min      5.003200e+04
25%      5.573810e+05
50%      8.762130e+05
75%      1.662643e+06
max      6.322586e+06
Name: total_students, dtype: float64


In [43]:
import numpy as np

merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table["log_enrollment"].describe())

count    861.000000
mean      13.701213
std        0.947233
min       10.820418
25%       13.231004
50%       13.683364
75%       14.323919
max       15.659639
Name: log_enrollment, dtype: float64


In [44]:
response = supabase.table("background_check_binary").select("*").execute()
bg_df = pd.DataFrame(response.data)
print(bg_df.head())

  state  year  background_check_law
0    MI  1994                     1
1    NE  2020                     1
2    KS  1998                     1
3    HI  1998                     1
4    CA  1980                     1


In [45]:
print(merged_df_table.columns.tolist())

['State', 'Year', 'incident_count', 'total_students', 'incident_rate_per_100k', 'log_enrollment']


In [46]:
incident_agg = (
    incident_df_from_1987
    .groupby(["State","Year"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

merged_df_table = enrollment_df.merge(
    incident_agg,
    on=["State","Year"],
    how="left"
)

merged_df_table["incident_count"] = merged_df_table["incident_count"].fillna(0)

print(merged_df_table.shape)
print((merged_df_table["incident_count"] == 0).sum())

(1989, 5)
1128


In [47]:
merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table.head())

        state  Year  total_students State  incident_count  log_enrollment
0  CALIFORNIA  2005         6322586    CA             6.0       15.659639
1  CALIFORNIA  2006         6312103    CA             4.0       15.657979
2  CALIFORNIA  2004         6298928    CA             7.0       15.655890
3  CALIFORNIA  2007         6274813    CA             5.0       15.652054
4  CALIFORNIA  2003         6244403    CA             4.0       15.647196


In [48]:
merged_df_table = merged_df_table.drop(columns=["state"])
print(merged_df_table.columns)

Index(['Year', 'total_students', 'State', 'incident_count', 'log_enrollment'], dtype='str')


In [49]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("background_check_binary")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

bg_df = pd.DataFrame(all_rows)

print(bg_df.shape)

(2295, 3)


In [50]:
policy_tables = [
    "background_check_binary",
    "permit_to_purchase_binary",
    "prohibited_possessor_binary",
    "waiting_period_binary",
    "registration_binary",
    "child_access_binary",
    "safety_training_binary",
    "report_stolen_lost_binary",
    "firearm_sales_retrictions_binary",
    "k12_settings_binary"
]

policy_dfs = {}

for table in policy_tables:
    all_rows = []
    start = 0
    batch_size = 1000

    while True:
        response = (
            supabase
            .table(table)
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
        )
        data = response.data
        if not data:
            break
        all_rows.extend(data)
        start += batch_size

    policy_dfs[table] = pd.DataFrame(all_rows)

print({k: v.shape for k, v in policy_dfs.items()})

{'background_check_binary': (2295, 3), 'permit_to_purchase_binary': (2115, 3), 'prohibited_possessor_binary': (2115, 3), 'waiting_period_binary': (2295, 3), 'registration_binary': (2295, 3), 'child_access_binary': (1620, 3), 'safety_training_binary': (405, 3), 'report_stolen_lost_binary': (765, 3), 'firearm_sales_retrictions_binary': (2295, 3), 'k12_settings_binary': (315, 3)}


In [51]:
for name, df in policy_dfs.items():
    merged_df_table = merged_df_table.merge(
        df,
        left_on=["State","Year"],
        right_on=["state","year"],
        how="left"
    )
    merged_df_table = merged_df_table.drop(columns=["state","year"])

# Replace NaN with 0 for all policy indicators
policy_cols = [col for col in merged_df_table.columns if col.endswith("_law")]
merged_df_table[policy_cols] = merged_df_table[policy_cols].fillna(0)

print(merged_df_table.shape)
print(len(policy_cols), "policy variables merged")

(1989, 15)
10 policy variables merged
